In [ ]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings

## Get spreadsheets

In [ ]:
# Function to normalize time between GFP, RFP, and OD
def split_time(obj):
    times_s = []
    for time in obj:
        hms = time.split(":")
        s = int(hms[0])*60+int(hms[1])
        times_s.append(s)
    return times_s

In [ ]:
# Fetch sheets and fix time
raw_od =  [pd.read_excel("../Data/Terminator_Strength-250423-1.xlsx", sheet_name="OD600"),
           pd.read_excel("../Data/Terminator_Strength-250423-2.xlsx", sheet_name="OD600"),
           pd.read_excel("../Data/Terminator_Strength-250423-3.xlsx", sheet_name="OD600")]
raw_gfp =  [pd.read_excel("../Data/Terminator_Strength-250423-1.xlsx", sheet_name="GFP"),
           pd.read_excel("../Data/Terminator_Strength-250423-2.xlsx", sheet_name="GFP"),
           pd.read_excel("../Data/Terminator_Strength-250423-3.xlsx", sheet_name="GFP")]
raw_rfp =  [pd.read_excel("../Data/Terminator_Strength-250423-1.xlsx", sheet_name="RFP"),
           pd.read_excel("../Data/Terminator_Strength-250423-2.xlsx", sheet_name="RFP"),
           pd.read_excel("../Data/Terminator_Strength-250423-3.xlsx", sheet_name="RFP")]
for df in raw_od:
    df["Time"] = split_time(df["Time"])
for df in raw_gfp:
    df["Time"] = split_time(df["Time"])
for df in raw_rfp:
    df["Time"] = split_time(df["Time"])

## Define samples

In [ ]:
# Function to combine two lists combinatorially
def combine(list1, list2, sep=""):
    new = []
    for el in list1:
        for el2 in list2:
            new.append(el+sep+el2)
    return(new)

In [ ]:
# Set working wells and samples
letters = ["A", "B", "C", "D", "E", "F", "G", "H"]
numbers = [str(num+1) for num in range(12)]
all_wells = combine(letters, numbers)
blank_well = "H12"

terminators = ["T7hyb1", "T7hyb8", "L3S2P11", "L3S1P00", "50bp"]
roadblocks = ["lacO", "tetO", "G3T", "G3Tr", "20bp", "20_lacO", "20_tetO", "20_G3T", "20_G3Tr"]
samples_reduced = combine(terminators, roadblocks, sep=".")
samples = combine(["0", "10"], samples_reduced, sep = "-")

In [ ]:
# Function to calculate mean and std from dfs
def raw_stats(df):
    mean_df = pd.DataFrame()
    mean_df.index = df.index
    mean_df["Blank"] = df[["G11", "G12", "H11", "H12"]].mean(axis=1)
    mean_df["AF"] = df[["A10", "A11", "A12"]].mean(axis=1)
    for idx, el in enumerate(samples):
        mean_df[el] = df[sample_wells[idx*3:idx*3+3]].mean(axis=1)
    
    
    std_df = pd.DataFrame()
    std_df.index = df.index
    std_df["Blank"] = df[["G11", "G12", "H11", "H12"]].std(axis=1)
    std_df["AF"] = df[["A10", "A11", "A12"]].std(axis=1)
    for idx, el in enumerate(samples):
        std_df[el] = df[sample_wells[idx*3:idx*3+3]].std(axis=1)
        #print(el, "->", sample_wells[idx*3:idx*3+3])

    return mean_df, std_df

In [ ]:
# Function to blank wells
def blank(df, std = False):
    blanked_df = pd.DataFrame()
    blanked_df.index = df.index
    if not std:
        blanked_df["AF"] = df["AF"] - df["Blank"]
        blanked_df[samples] = df[samples].sub(df["Blank"], axis=0)
    else:
        blanked_df["AF"] = np.sqrt(df["AF"]**2 + df["Blank"]**2)
        blanked_df[samples] = np.sqrt(np.power(df[samples], 2).add(np.power(df["Blank"], 2), axis=0))

    return blanked_df

In [ ]:
# Function to remove AF
def remove_af(df, std = False):
    blanked_df = pd.DataFrame()
    blanked_df.index = df.index
    if not std:
        blanked_df[samples] = df[samples].sub(df["AF"], axis=0)
    else:
        blanked_df[samples] = np.sqrt(np.power(df[samples], 2).add(np.power(df["AF"], 2), axis=0))

    return blanked_df

## Plot all curves

In [ ]:
fig, axs = plt.subplots(8, 12, figsize = (40, 20), constrained_layout=True)
fig.suptitle("RAW_OD600")
for r in range(3):
    df = raw_od[r]
    for i in range(96):
        axs[i//12, i%12].plot(df.index, df[all_wells[i]], color='grey')
        axs[i//12, i%12].set_ylim(10**-1, 10**0)
        axs[i//12, i%12].set_xlim(0, df.index.max())
        axs[i//12, i%12].set_yscale('log')
plt.show()

In [ ]:
fig, axs = plt.subplots(8, 12, figsize = (40, 20), constrained_layout=True)
fig.suptitle("RAW_sfGFP")
for r in range(3):
    df = raw_gfp[r]
    for i in range(96):
        axs[i//12, i%12].plot(df.index, df[all_wells[i]], color='green')
        axs[i//12, i%12].set_ylim(10**4, 10**7)
        axs[i//12, i%12].set_xlim(0, df.index.max())
        axs[i//12, i%12].set_yscale('log')
plt.show()

In [ ]:
fig, axs = plt.subplots(8, 12, figsize = (40, 20), constrained_layout=True)
fig.suptitle("RAW_mScarlet")
for r in range(3):
    df = raw_rfp[r]
    for i in range(96):
        axs[i//12, i%12].plot(df.index, df[all_wells[i]], color='red')
        axs[i//12, i%12].set_ylim(10**3, 10**7)
        axs[i//12, i%12].set_xlim(0, df.index.max())
        axs[i//12, i%12].set_yscale('log')
plt.show()

## Calculate time of max growth

In [ ]:
diff_od = raw_od[2].iloc[1:].reset_index() - raw_od[0].iloc[:-1]
diff_od = diff_od.drop(columns = ["Time", "Temperature(¡C)", "index"])
diff_od["Time"] = raw_od[0]["Time"][1:].to_list()
diff_od = diff_od.set_index("Time")

time4max = {}
for col in diff_od.columns:
    time4max[col] = diff_od[col].idxmax()
    

## Delete non-growing wells

In [ ]:

delete_wells = ["A5", "A6", "B2", "B10", "C6", "C12", "E5", "E6", "F2", "F10", "G6", "G12"]
for i in range(3):
    raw_od[i][delete_wells] = np.nan
    raw_gfp[i][delete_wells] = np.nan
    raw_rfp[i][delete_wells] = np.nan

    raw_od[i] = raw_od[i].set_index('Time')
    raw_gfp[i] = raw_gfp[i].set_index('Time')
    raw_rfp[i] = raw_rfp[i].set_index('Time')

    raw_od[i] = raw_od[i].drop(columns = 'Temperature(¡C)')
    raw_gfp[i] = raw_gfp[i].drop(columns = 'Temperature(¡C)')
    raw_rfp[i] = raw_rfp[i].drop(columns = 'Temperature(¡C)')


In [ ]:
delete_0 = ["E3"]
raw_od[0][delete_0] = np.nan
raw_gfp[0][delete_0] = np.nan
raw_rfp[0][delete_0] = np.nan

delete_1 = ["A3", "C9", "C10"]
raw_od[1][delete_1] = np.nan
raw_gfp[1][delete_1] = np.nan
raw_rfp[1][delete_1] = np.nan

## Take only last time point (6h experiment only)

In [ ]:
last_od = []
last_gfp = []
last_rfp = []

for i in range(3):
    last_od.append(raw_od[i].iloc[-2])
    last_gfp.append(raw_gfp[i].iloc[-2])
    last_rfp.append(raw_rfp[i].iloc[-2])


## Calculate mean, std for RPU, AF, Blank

In [ ]:
rpu_wells = ["D10", "H10"]
af_wells = ["D11", "H11"]
blk_wells = ["D12", "H12"]

In [ ]:
consts_od = pd.DataFrame(columns = ["RPU", "AF", "Blank"])
consts_gfp = pd.DataFrame(columns = ["RPU", "AF", "Blank"])
consts_rfp = pd.DataFrame(columns = ["RPU", "AF", "Blank"])

consts_od["RPU"] = last_od[0][rpu_wells].to_list() + last_od[1][rpu_wells].to_list() + last_od[2][rpu_wells].to_list()
consts_gfp["RPU"] = last_gfp[0][rpu_wells].to_list() + last_gfp[1][rpu_wells].to_list() + last_gfp[2][rpu_wells].to_list()
consts_rfp["RPU"] = last_rfp[0][rpu_wells].to_list() + last_rfp[1][rpu_wells].to_list() + last_rfp[2][rpu_wells].to_list()

consts_od["AF"] = last_od[0][af_wells].to_list() + last_od[1][af_wells].to_list() + last_od[2][af_wells].to_list()
consts_gfp["AF"] = last_gfp[0][af_wells].to_list() + last_gfp[1][af_wells].to_list() + last_gfp[2][af_wells].to_list()
consts_rfp["AF"] = last_rfp[0][af_wells].to_list() + last_rfp[1][af_wells].to_list() + last_rfp[2][af_wells].to_list()

consts_od["Blank"] = last_od[0][blk_wells].to_list() + last_od[1][blk_wells].to_list() + last_od[2][blk_wells].to_list()
consts_gfp["Blank"] = last_gfp[0][blk_wells].to_list() + last_gfp[1][blk_wells].to_list() + last_gfp[2][blk_wells].to_list()
consts_rfp["Blank"] = last_rfp[0][blk_wells].to_list() + last_rfp[1][blk_wells].to_list() + last_rfp[2][blk_wells].to_list()

## Blank, divide OD, and remove AF from RPU

In [ ]:
consts_od_mean = consts_od.mean()
consts_gfp_mean = consts_gfp.mean()
consts_rfp_mean = consts_rfp.mean()

consts_od_std = consts_od.std()
consts_gfp_std = consts_gfp.std()
consts_rfp_std = consts_rfp.std()

for df in [consts_od_mean, consts_gfp_mean, consts_rfp_mean]:
    df["RPU"] = df["RPU"] - df["Blank"]
    df["AF"] = df["AF"] - df["Blank"]
    
for df in [consts_od_std, consts_gfp_std, consts_rfp_std]:
    df["RPU"] = np.sqrt(df["RPU"]**2 + df["Blank"]**2)
    df["AF"] = np.sqrt(df["AF"]**2 + df["Blank"]**2)

In [ ]:
for pair in [[consts_gfp_mean, consts_gfp_std], [consts_rfp_mean, consts_rfp_std]]:
    save_rpu =  pair[0]["RPU"]
    save_af = pair[0]["AF"]
    
    pair[0]["RPU"] = pair[0]["RPU"] / consts_od_mean["RPU"]
    pair[0]["AF"] = pair[0]["AF"] / consts_od_mean["AF"]
    
    pair[1]["RPU"] = pair[0]["RPU"] * np.sqrt((pair[1]["RPU"]/save_rpu)**2 + (consts_od_std["RPU"]/consts_od_mean["RPU"])**2)
    pair[1]["AF"] = pair[0]["AF"] * np.sqrt((pair[1]["AF"]/save_af)**2 + (consts_od_std["AF"]/consts_od_mean["AF"])**2)

for df in [consts_gfp_mean, consts_rfp_mean]:
    df["RPU"] = df["RPU"] - df["AF"]

for df in [consts_gfp_std, consts_rfp_std]:
    df["RPU"] = np.sqrt(df["RPU"]**2 + df["AF"]**2)

In [ ]:
print(consts_gfp_mean)
print(consts_gfp_std)

## Drop RPU, AF, Blank wells from data

In [ ]:
for triple in [last_od, last_gfp, last_rfp]:
    for i in range(3):
        triple[i] = triple[i].drop(rpu_wells + af_wells + blk_wells)

## Calculate data mean and std

In [ ]:
mean_od = pd.DataFrame()
mean_gfp = pd.DataFrame()
mean_rfp = pd.DataFrame()

std_od = pd.DataFrame()
std_gfp = pd.DataFrame()
std_rfp = pd.DataFrame()

# Ignore warnings because we'll be taking means of nan lists (missing data)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    for col in last_od[0].keys():
        mean_od[col] = [np.nanmean([last_od[0][col], last_od[1][col], last_od[2][col]])]
        mean_gfp[col] = [np.nanmean([last_gfp[0][col], last_gfp[1][col], last_gfp[2][col]])]
        mean_rfp[col] = [np.nanmean([last_rfp[0][col], last_rfp[1][col], last_rfp[2][col]])]
        
        std_od[col] = [np.nanstd([last_od[0][col], last_od[1][col], last_od[2][col]])]
        std_gfp[col] = [np.nanstd([last_gfp[0][col], last_gfp[1][col], last_gfp[2][col]])]
        std_rfp[col] = [np.nanstd([last_rfp[0][col], last_rfp[1][col], last_rfp[2][col]])]

In [ ]:
mean_od.columns = samples
mean_gfp.columns = samples
mean_rfp.columns = samples

std_od.columns = samples
std_gfp.columns = samples
std_rfp.columns = samples

## Divide by OD and remove AF

In [ ]:
blanked_mean_od = mean_od - consts_od_mean["Blank"]
blanked_mean_gfp = mean_gfp - consts_gfp_mean["Blank"]
blanked_mean_rfp = mean_rfp - consts_rfp_mean["Blank"]

blanked_std_od = np.sqrt(np.power(std_od, 2) + consts_od_std["Blank"]**2)
blanked_std_gfp = np.sqrt(np.power(std_gfp, 2) + consts_gfp_std["Blank"]**2)
blanked_std_rfp = np.sqrt(np.power(std_rfp, 2) + consts_rfp_std["Blank"]**2)

In [ ]:
mean_G_OD = blanked_mean_gfp/blanked_mean_od
std_G_OD = abs(mean_G_OD) * np.sqrt((blanked_std_gfp/blanked_mean_gfp)**2 + (blanked_std_od/blanked_mean_od)**2)

mean_R_OD = blanked_mean_rfp/blanked_mean_od
std_R_OD = abs(mean_R_OD) * np.sqrt((blanked_std_rfp/blanked_mean_rfp)**2 + (blanked_std_od/blanked_mean_od)**2)

In [ ]:
mean_G_OD_AF = mean_G_OD - consts_gfp_mean["AF"]
mean_R_OD_AF = mean_R_OD - consts_rfp_mean["AF"]

std_G_OD_AF = np.sqrt(np.power(std_G_OD, 2) + consts_gfp_std["AF"]**2)
std_R_OD_AF = np.sqrt(np.power(std_R_OD, 2) + consts_rfp_std["AF"]**2)

## Plot base GFP and RFP

In [ ]:
fig, axs = plt.subplots(5, 9, figsize = (20, 10), constrained_layout = True)
for i in range(45):
    axs[i//9, i%9].bar([1, 2], [mean_G_OD_AF.iloc[0].iloc[i], mean_G_OD_AF.iloc[0].iloc[i+45]], color="g")
    axs[i//9, i%9].errorbar([1, 2],
                            [mean_G_OD_AF.iloc[0].iloc[i], mean_G_OD_AF.iloc[0].iloc[i+45]],
                            yerr = [std_G_OD_AF.iloc[0].iloc[i], std_G_OD_AF.iloc[0].iloc[i+45]],
                            marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
    axs[i//9, i%9].bar([3, 4], [mean_R_OD_AF.iloc[0].iloc[i], mean_R_OD_AF.iloc[0].iloc[i+45]], color="r")
    axs[i//9, i%9].errorbar([3, 4],
                            [mean_R_OD_AF.iloc[0].iloc[i], mean_R_OD_AF.iloc[0].iloc[i+45]],
                            yerr = [std_R_OD_AF.iloc[0].iloc[i], std_R_OD_AF.iloc[0].iloc[i+45]],
                            marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
    axs[i//9, i%9].set_yscale("log")
    axs[i//9, i%9].set_ylim(10**4, 10**9)
    axs[i//9, i%9].set_xlim(0.2, 4.8)
    axs[i//9, i%9].set_xticks([1.5, 3.5])
    axs[i//9, i%9].set_xticklabels(["sfGFP", "mScarlet"])
    if i%9 == 0:
        axs[i//9, i%9].set(ylabel = samples_reduced[i].split(".")[0])
    if i//9 == 0:
        axs[i//9, i%9].set_title(samples_reduced[i].split(".")[1])
    if i in [4, 5, 13, 21, 29, 35]:
        axs[i//9, i%9].text(2.5, 10**6, "NO DATA", ha="center")
    else:
        axs[i//9, i%9].grid()
        axs[i//9, i%9].set_axisbelow(True)
plt.savefig("library_RandG.png")
plt.show()

## Calculate fluorescence above uninduced levels and red/green ratio

In [ ]:
Fluos_mean_IU = []
Fluos_std_IU = []
for df in [mean_G_OD_AF, mean_R_OD_AF]:
    uninduced = df[df.columns[:45]]
    induced = df[df.columns[-45:]]
    uninduced.columns = samples_reduced
    induced.columns = samples_reduced
    Fluos_mean_IU.append(induced - uninduced)

for df in [std_G_OD_AF, std_R_OD_AF]:
    uninduced = df[df.columns[:45]]
    induced = df[df.columns[-45:]]
    uninduced.columns = samples_reduced
    induced.columns = samples_reduced
    Fluos_std_IU.append(np.sqrt(induced**2 + uninduced**2))

mean_G_IU, mean_R_IU = Fluos_mean_IU
std_G_IU, std_R_IU = Fluos_std_IU
    

In [ ]:
fig, axs = plt.subplots(5, 9, figsize = (20, 10), constrained_layout = True)
for i in range(45):
    axs[i//9, i%9].bar([1], [mean_G_IU.iloc[0].iloc[i]], color="green")
    axs[i//9, i%9].errorbar([1],
                            [mean_G_IU.iloc[0].iloc[i]],
                            yerr = [std_G_IU.iloc[0].iloc[i]],
                            marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
    axs[i//9, i%9].bar([2], [mean_R_IU.iloc[0].iloc[i]], color="red")
    axs[i//9, i%9].errorbar([2],
                            [mean_R_IU.iloc[0].iloc[i]],
                            yerr = [std_R_IU.iloc[0].iloc[i]],
                            marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
    axs[i//9, i%9].set_yscale("log")
    axs[i//9, i%9].set_ylim(10**4, 10**9)
    axs[i//9, i%9].set_xlim(0.2, 2.8)
    if i%9 == 0:
        axs[i//9, i%9].set(ylabel = samples_reduced[i].split(".")[0])
    if i//9 == 0:
        axs[i//9, i%9].set_title(samples_reduced[i].split(".")[1])
    if i in [4, 5, 13, 21, 29, 35]:
        axs[i//9, i%9].text(1.5, 10**6, "NO DATA", ha="center")
        
plt.show()

In [ ]:
mean_RG = (mean_R_IU*1000)/mean_G_IU
std_RG = mean_RG * np.sqrt(((std_R_IU*1000)/(mean_R_IU*1000))**2 + (std_G_IU/mean_G_IU)**2)

mean_rel_RG = mean_RG / mean_RG["50bp.20bp"].squeeze()
std_rel_RG = mean_rel_RG * np.sqrt((std_RG/mean_RG)**2 + (std_RG["50bp.20bp"].squeeze()/mean_RG["50bp.20bp"].squeeze())**2)

In [ ]:
fig, axs = plt.subplots(5, 1, figsize = (8, 10), constrained_layout= True)
for i in range(5):
    x_labels = roadblocks
    axs[i].bar(x_labels, mean_RG[samples_reduced[i*9:i*9+9]].iloc[0], color = "y")
    axs[i].errorbar(x_labels,
                    mean_RG[samples_reduced[i*9:i*9+9]].iloc[0],
                    yerr = std_RG[samples_reduced[i*9:i*9+9]].iloc[0],
                    marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
    axs[i].set(ylabel = terminators[i]+" R/G")
    #low_bound = int(min([10*np.floor(min(mean_RG[samples_reduced[i*9:i*9+9]].iloc[0])//10), 0]))
    #axs[i].set_ylim(bottom=low_bound, top=100)
    axs[i].set_yscale("log")
    axs[i].set_ylim(10**1, 10**4)
    axs[i].set_xlim(-0.7, 8.7)
    axs[i].set_xticks(range(9))
    #axcs[i].set_xticklabels(xlabels)
    axs[i].grid()
    axs[i].set_axisbelow(True)
    axs[i].set_yticks(np.linspace(low_bound, 100, (100-low_bound)//10 + 1))
    for j in range(9):
        if np.isnan(mean_RG.iloc[0].to_list()[i*9 + j]):
            axs[i].text(j, 50, "NO DATA", rotation = 90, va="center", ha="center")
plt.savefig("library_RG_bars.png")
plt.show()

## Calculate Termination Efficiency

In [ ]:
mean_Te = 100 * (1 - mean_rel_RG)
std_Te = std_rel_RG * 100
std_Te["50bp.20bp"] = np.nan

In [ ]:
std_RG

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(samples_reduced, mean_Te.iloc[0], color = "gray")
'''
ax.errorbar(samples_reduced,
            mean_Te.iloc[0],
            yerr = std_Te.iloc[0],
            marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
'''
ax.set_xticklabels(samples_reduced, rotation=45, ha="right", rotation_mode="anchor")
plt.show()

In [ ]:
fig, axs = plt.subplots(5, 1, figsize = (8, 10), constrained_layout= True)
for i in range(5):
    x_labels = roadblocks
    axs[i].bar(x_labels, mean_Te[samples_reduced[i*9:i*9+9]].iloc[0], color = "gray")
    axs[i].errorbar(x_labels,
                    mean_Te[samples_reduced[i*9:i*9+9]].iloc[0],
                    yerr = std_Te[samples_reduced[i*9:i*9+9]].iloc[0],
                    marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
    axs[i].set(ylabel = terminators[i]+" TE (%)")
    low_bound = int(min([10*np.floor(min(mean_Te[samples_reduced[i*9:i*9+9]].iloc[0])//10), 0]))
    axs[i].set_ylim(bottom=low_bound, top=100)
    axs[i].set_xlim(-0.7, 8.7)
    axs[i].set_xticks(range(9))
    #axcs[i].set_xticklabels(xlabels)
    axs[i].grid()
    axs[i].set_axisbelow(True)
    axs[i].set_yticks(np.linspace(low_bound, 100, (100-low_bound)//10 + 1))
    for j in range(9):
        if np.isnan(mean_Te.iloc[0].to_list()[i*9 + j]):
            axs[i].text(j, 50, "NO DATA", rotation = 90, va="center", ha="center")
        if i == 4 and j == 4:
            axs[i].text(j, 50, "REFERENCE", rotation = 90, va="center", ha="center")
plt.savefig("library_Te_bars.png")
plt.show()

In [ ]:
mean_Te_2D = []
for i in range(5):
    mean_Te_2D.append(mean_Te.iloc[0].to_list()[i*9:i*9+9])
mean_Te_2D = np.array(mean_Te_2D)

In [ ]:
fig, ax = plt.subplots(constrained_layout=True)
im = ax.imshow(mean_Te_2D, cmap="Reds_r")
terminators = ["T7hyb1", "T7hyb8", "L3S2P11", "L3S1P00", "50bp"]
roadblocks = ["lacO", "tetO", "G3T", "G3Tr", "20bp", "20_lacO", "20_tetO", "20_G3T", "20_G3Tr"]
ax.set_xticks(range(len(roadblocks)), labels=roadblocks,
              rotation=45, ha="right", rotation_mode="anchor")
ax.set_yticks(range(len(terminators)), labels=terminators)
for i in range(len(terminators)):
    for j in range(len(roadblocks)):
        text = ax.text(j, i, mean_Te_2D[i, j].round(1),
                       ha="center", va="center", color="k")
ax.set_title("Terminator Efficiency (%)")

plt.savefig("library_Te.png")
plt.show()

In [ ]:
Te_min = []
for j in range(5):
    Te_set = mean_Te[samples_reduced[j*9:j*9+9]].iloc[0].to_list()
    #if np.isnan(Te_set[4]):
    #    Te_min.append(min(Te_set))
    #else:
    #    Te_min.append(Te_set[4])
    Te_min.append(Te_set[4])

mean_rb_change = []
std_rb_change = []
for i in range(9):
    name = roadblocks[i]
    rb_set = mean_Te[samples_reduced[i::9]].iloc[0].to_list()
    diff_from_20bp = 100*(np.array(rb_set) - np.array(Te_min)) / (100 - np.array(Te_min))
    print(name, diff_from_20bp)
    mean_rb_change.append(np.nanmean(diff_from_20bp))
    std_rb_change.append(np.nanstd(diff_from_20bp))

In [ ]:
fig, ax = plt.subplots(figsize = (6, 3))
ax.bar(roadblocks, mean_rb_change)
ax.errorbar(roadblocks, mean_rb_change, yerr = std_rb_change,
                    marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
ax.set_xticks(range(len(roadblocks)), labels=roadblocks,
              rotation=45, ha="right", rotation_mode="anchor")
ax.set_yticks(np.linspace(-20, 100, 13))
ax.set_ylabel("TE increase (%)")
ax.grid()
ax.set_axisbelow(True)
plt.show()